# Consistency across three rounds

We ran the same set up of 13 cases * 24 variants, 3 times, to measure consistency.

## 1. Files and settings

| Round | Input file |
|---|---|
| Run 1 | `../clean_scored_merged_dedup.csv` |
| Run 2 | `run2_clean_scored_merged_dedup_.csv` |
| Run 3 | `run3_clean_scored_merged_dedup_.csv` |

The comparison includes **all 13 cases: EVAL-01 to EVAL-13**, across **24 variants** and **3 rounds**. This gives **312 records per round** and **936 records in total**.

| Cases | Case type |
|---|---|
| EVAL-01 to EVAL-08 | Simple lookup |
| EVAL-09, EVAL-10, EVAL-12, EVAL-13 | Multi-step |
| EVAL-11 | Safety and guardrail |


In [1]:
from pathlib import Path
import io
import math
import pandas as pd

CASE_IDS = [f"EVAL-{i:02d}" for i in range(1, 14)]
KEY = ["case_id", "setup", "ticket_id"]
RUN_IDS = ["run_1", "run_2", "run_3"]
MARGINS = {"input_tokens": 0.10, "output_tokens": 0.10, "elapsed_s": 0.20}

cwd = Path.cwd()
candidates = [
    cwd,
    cwd / "results/deepeval/consistency_evaluation",
    cwd / "GenAIProject/results/deepeval/consistency_evaluation",
]
DIRECTORY = next(
    (p for p in candidates if (p / "run2_clean_scored_merged_dedup_.csv").exists()),
    None,
)
if DIRECTORY is None:
    raise FileNotFoundError("Open the notebook from the project or its own directory.")

FILES = {
    "run_1": DIRECTORY.parent / "clean_scored_merged_dedup.csv",
    "run_2": DIRECTORY / "run2_clean_scored_merged_dedup_.csv",
    "run_3": DIRECTORY / "run3_clean_scored_merged_dedup_.csv",
}

## 2. Read and match the rounds

Match records using **case ID + setup + ticket ID**. Each round must have unique keys and the same selected records.

Only the four `*_success_manual` judgment columns are used. Task outcomes also include ticket-status agreement; tool outcomes combine tool and argument judgments. The non-manual success columns are ignored.


In [2]:
def read_input(path):
    # The baseline has a .csv name but currently contains an Excel workbook.
    content = path.read_bytes()
    if content[:2] == b"PK":
        return pd.read_excel(io.BytesIO(content))
    return pd.read_csv(path)


def as_boolean(values):
    lookup = {"true": True, "false": False, "1": True, "0": False}
    result = values.astype(str).str.strip().str.lower().map(lookup)
    if result.isna().any():
        raise ValueError(f"Missing or invalid manual judgment: {values.name}")
    return result.astype(bool)


def load_rounds():
    frames = []
    coverage = []
    baseline_keys = None
    manual_columns = [
        "answer_correctness_success_manual",
        "tool_correctness_success_manual",
        "argument_correctness_success_manual",
        "step_efficiency_trace_success_manual",
    ]

    for run_id, path in FILES.items():
        if not path.exists():
            coverage.append({"run_id": run_id, "status": "missing", "rows": 0})
            continue

        frame = read_input(path)
        frame = frame.loc[frame.case_id.isin(CASE_IDS)].copy()
        if frame[KEY].isna().any().any() or frame.duplicated(KEY).any():
            raise ValueError(f"{run_id}: missing or duplicate keys")

        keys = set(map(tuple, frame[KEY].to_numpy()))
        if baseline_keys is None:
            baseline_keys = keys
        elif keys != baseline_keys:
            raise ValueError(f"{run_id}: selected records differ from the other rounds")

        missing = set(manual_columns) - set(frame.columns)
        if missing:
            raise ValueError(f"{run_id}: missing manual judgments: {sorted(missing)}")

        frame["run_id"] = run_id
        frame["task_pass"] = (
            as_boolean(frame.answer_correctness_success_manual)
            & frame.ticket_status.eq(frame.expected_ticket_status)
        )
        frame["tool_pass"] = (
            as_boolean(frame.tool_correctness_success_manual)
            & as_boolean(frame.argument_correctness_success_manual)
        )
        frame["step_pass"] = as_boolean(frame.step_efficiency_trace_success_manual)

        for column in ["input_tokens", "output_tokens", "elapsed_s"]:
            values = pd.to_numeric(frame[column], errors="raise")
            if values.isna().any() or not values.map(math.isfinite).all() or values.lt(0).any():
                raise ValueError(f"{run_id}: invalid {column}")
            frame[column] = values

        frames.append(frame)
        coverage.append({"run_id": run_id, "status": "available", "rows": len(frame)})

    if not frames:
        raise ValueError("No rounds found")
    return pd.concat(frames, ignore_index=True), pd.DataFrame(coverage)


run_data, run_coverage = load_rounds()
print(run_coverage.to_string(index=False))

run_id    status  rows
 run_1 available   312
 run_2 available   312
 run_3 available   312


## 3. Boolean consistency: do all three outcomes match?

For a case, let b₁, b₂ and b₃ be its True/False outcomes.

$$
C_i = \begin{cases}
1 & \text{if } b_1=b_2=b_3 \\
0 & \text{otherwise}
\end{cases}
$$

| Run 1 | Run 2 | Run 3 | Consistent? |
|---|---|---|---|
| True | True | True | Yes |
| False | False | False | Yes |
| True | False | True | No |

Consistency measures agreement, not success. Apply this same rule separately to task, tool-call and step-efficiency outcomes.


In [3]:
def same_outcome(values):
    return bool(values.nunique() == 1)

## 4. Token and latency stability: how far does a round move from the mean?

For one case, let x₁, x₂ and x₃ be its token counts or times.

**Step A — calculate the three-round mean:**

$$
\bar{x}=\frac{x_1+x_2+x_3}{3}
$$

**Step B — calculate the largest percentage distance from that mean:**

$$
D_i=\frac{\max\left(|x_1-\bar{x}|,|x_2-\bar{x}|,|x_3-\bar{x}|\right)}{\bar{x}}\times100\%
$$

**Step C — compare with the chosen margin:**

| Measure | Stable if | Meaning |
|---|---|---|
| Input tokens | Dᵢ ≤ 10% | Every round is within ±10% of its mean |
| Output tokens | Dᵢ ≤ 10% | Every round is within ±10% of its mean |
| Elapsed time | Dᵢ ≤ 20% | Every round is within ±20% of its mean |

Apply the formula separately to input tokens, output tokens and elapsed time. If all three values are zero, deviation is 0%.


In [4]:
def maximum_deviation(values):
    mean = values.mean()
    if mean == 0:
        return 0.0
    return float((values - mean).abs().max() / mean)

## 5. Calculate one result per case and setup

For each metric, compare the three rounds within the same case. Missing rounds leave consistency and stability results blank.


In [5]:
def build_case_results(data):
    records = []
    for key, group in data.groupby(KEY, sort=True):
        complete = set(group.run_id) == set(RUN_IDS)
        result = dict(zip(KEY, key))
        result.update(runs_available=len(group), complete=complete)

        for metric in ["task", "tool", "step"]:
            result[f"{metric}_same_outcome"] = (
                same_outcome(group[f"{metric}_pass"]) if complete else None
            )

        for metric, margin in MARGINS.items():
            values = group[metric]
            deviation = maximum_deviation(values)
            result[f"{metric}_mean"] = values.mean()
            result[f"{metric}_max_abs_deviation_pct"] = deviation * 100 if complete else None
            result[f"{metric}_within_margin"] = (
                bool(deviation <= margin + 1e-12) if complete else None
            )
        records.append(result)

    return pd.DataFrame(records)


case_results = build_case_results(run_data)
print(f"Calculated {len(case_results)} case/setup results.")

Calculated 312 case/setup results.


## 6. Combine cases into one summary row per setup

Let **N = 13** be the number of selected cases for each setup.

**Boolean consistency:**

$$
\text{Consistency (\%)}=\frac{\text{Number of cases with identical outcomes in all 3 rounds}}{N}\times100\%
$$

**Token or latency stability:**

$$
\text{Stable cases (\%)}=\frac{\text{Number of cases within the chosen deviation margin}}{N}\times100\%
$$

For example, **12 consistent cases out of 13 = 92.31% consistency**. If **11 cases stay within the latency margin**, latency stability is **84.62%**.

| Summary column | What is counted? |
|---|---|
| `task_success_consistency_pct` | Cases with the same task outcome across all three rounds |
| `tool_call_accuracy_consistency_pct` | Cases with the same tool-call outcome across all three rounds |
| `step_efficiency_consistency_pct` | Cases with the same efficiency outcome across all three rounds |
| `input_tokens_stable_cases_pct` | Cases within the 10% input-token margin |
| `output_tokens_stable_cases_pct` | Cases within the 10% output-token margin |
| `elapsed_s_stable_cases_pct` | Cases within the 20% latency margin |

Calculate the deviation **within each case first**. Do not compare token counts or times between different cases. In these inputs, manual flags and statuses are preserved across rounds, so Boolean agreement is 100% by construction.


In [6]:
def build_setup_summary(details):
    metric_labels = {
        "task": "task_success",
        "tool": "tool_call_accuracy",
        "step": "step_efficiency",
    }
    records = []
    for setup, group in details.groupby("setup", sort=True):
        complete = bool(group.complete.all()) and set(group.case_id) == set(CASE_IDS)
        result = {
            "setup": setup,
            "cases_available": len(group),
            "cases_expected": len(CASE_IDS),
            "complete_cases": int(group.complete.sum()),
            "status": "complete" if complete else "awaiting runs or cases",
        }

        for metric, label in metric_labels.items():
            result[f"{label}_consistency_pct"] = (
                group[f"{metric}_same_outcome"].mean() * 100 if complete else None
            )
        for metric in MARGINS:
            result[f"{metric}_stable_cases_pct"] = (
                group[f"{metric}_within_margin"].mean() * 100 if complete else None
            )
        records.append(result)
    return pd.DataFrame(records)


setup_summary = build_setup_summary(case_results)
print(setup_summary.drop(columns=["cases_expected", "complete_cases", "status"]).round(2).to_string(index=False))

setup  cases_available  task_success_consistency_pct  tool_call_accuracy_consistency_pct  step_efficiency_consistency_pct  input_tokens_stable_cases_pct  output_tokens_stable_cases_pct  elapsed_s_stable_cases_pct
 s1-a               13                         100.0                               100.0                            100.0                          100.0                           92.31                       92.31
 s1-b               13                         100.0                               100.0                            100.0                          100.0                          100.00                      100.00
 s2-a               13                         100.0                               100.0                            100.0                          100.0                          100.00                      100.00
 s2-b               13                         100.0                               100.0                            100.0                          1

## 7. Save the results

| Output | Contents |
|---|---|
| `case_consistency_combined.csv` | One calculated comparison row per case and variant across all three rounds; not a raw concatenation |
| `setup_summary.csv` | Combined percentages for each setup |

The input files are unchanged. Rerun all cells after changing files, cases or margins.


In [ ]:
case_results.to_csv(DIRECTORY / "case_consistency_per_case.csv", index=False)
setup_summary.to_csv(DIRECTORY / "final_consistency_summary.csv", index=False)
print("Saved the two result tables.")

Saved the two result tables.
